# 02. Data Preparation

This notebook will be used to prepare the input data for the ClearCNV model. We will load the `SpatialData` object, explore it, and then prepare the three required inputs for the model:
1. Gene expression matrix
2. Spatial graph
3. Gene bins

In [1]:
import os
import spatialdata as sd
import numpy as np
import geopandas as gpd
from sklearn.neighbors import kneighbors_graph
import torch
import infercnvpy as cnv
import scanpy as sc
import scipy.sparse

/usr/lib/python3.12/importlib/__init__.py:90: FutureWarning: The legacy Dask DataFrame implementation is deprecated and will be removed in a future version. Set the configuration option `dataframe.query-planning` to `True` or None to enable the new Dask Dataframe implementation and silence this warning.
  return _bootstrap._gcd_import(name[level:], package, level)
/usr/local/lib/python3.12/dist-packages/cudf/utils/gpu_utils.py:75: UserWarning: Failed to dlopen libcuda.so.1
  warnings.warn(str(e))


## 1. Load the SpatialData object

In [2]:
data_dir = "../datasets"
sdata_path = os.path.join(data_dir, "Xenium5K_human_prostate.zarr")
sdata = sd.read_zarr(sdata_path)

/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)


/usr/local/lib/python3.12/dist-packages/zarr/creation.py:610: UserWarning: ignoring keyword argument 'read_only'
  compressor, fill_value = _kwargs_compat(compressor, fill_value, kwargs)
/usr/lib/python3.12/functools.py:912: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [3]:
import numpy as np
import geopandas as gpd
from shapely.geometry import Point

def adata_cell_coords_from_sdata(
    sdata,
    table_key="table",
    prefer_obsm_key="spatial",
    crs_like_shapes=True
):
    """
    Build a GeoDataFrame of cell centroids from the AnnData table inside SpatialData.
    Tries obsm['spatial'] first, then common obs columns.
    """
    adata = sdata.tables[table_key]

    # 1) Try obsm['spatial'] (standard convention)
    coords = None
    if prefer_obsm_key in adata.obsm:
        arr = np.asarray(adata.obsm[prefer_obsm_key])
        if arr.shape[1] >= 2:
            coords = arr[:, :2]

    # 2) Fall back to common obs column names
    if coords is None:
        candidates = [("x", "y"), ("X", "Y"), ("px_x", "px_y"), ("cx", "cy")]
        for xk, yk in candidates:
            if xk in adata.obs.columns and yk in adata.obs.columns:
                coords = adata.obs[[xk, yk]].to_numpy()
                break

    if coords is None:
        raise KeyError(
            "Could not find coordinates in adata. "
            "Expected obsm['spatial'] or obs columns like ('x','y')."
        )

    # 3) Make GeoDataFrame with points
    gdf = gpd.GeoDataFrame(
        {"x": coords[:, 0], "y": coords[:, 1]},
        index=adata.obs_names,
        geometry=[Point(xy) for xy in coords],
    )
    gdf.index.name = "cell_id"

    # 4) (Optional) adopt CRS from shapes to ensure overlay consistency
    if crs_like_shapes and "cell_boundaries" in sdata.shapes:
        shapes_gdf = sdata.shapes["cell_boundaries"]
        try:
            gdf.set_crs(shapes_gdf.crs, inplace=True)
        except Exception:
            # If shapes have no CRS, leave as-is
            pass

    return gdf

In [4]:

# Usage:
cell_coords_adata = adata_cell_coords_from_sdata(sdata)
cell_coords_adata.head()

,x,y,geometry
cell_id,,,
0,170.855087,2017.241211,POINT (170.855 2017.241)
1,141.605698,2481.442139,POINT (141.606 2481.442)
2,198.382446,2414.510010,POINT (198.382 2414.51)
3,129.322784,2834.657227,POINT (129.323 2834.657)
4,649.650635,2995.655762,POINT (649.651 2995.656)


In [5]:
adata = sdata.tables["table"]

# Align by cell IDs to ensure matching order
adata.obs["x"] = cell_coords_adata.loc[adata.obs_names, "x"].values
adata.obs["y"] = cell_coords_adata.loc[adata.obs_names, "y"].values

#### Downsample the dataset (optional)

In [7]:
# Set random seed for reproducibility
np.random.seed(42)

# Extract the table (AnnData) from the SpatialData object
adata = sdata.tables["table"]  # or the correct key if different
print("Original number of cells:", adata.n_obs)

# Compute number of cells to keep (5%)
n_cells = int(adata.n_obs * 0.05)

# Randomly select indices
selected_idx = np.random.choice(adata.n_obs, n_cells, replace=False)

# Subset the AnnData table
adata_subset = adata[selected_idx, :].copy()

# Create a new SpatialData object with the subsetted table
sdata_subset = sd.SpatialData(
    tables={"table": adata_subset},
    # optionally keep same images, points, or labels
    images=sdata.images if hasattr(sdata, "images") else None,
    labels=sdata.labels if hasattr(sdata, "labels") else None,
    points=sdata.points if hasattr(sdata, "points") else None,
)

print("Downsampled to:", sdata_subset.tables["table"].n_obs, "cells")


Original number of cells: 193000
Downsampled to: 9650 cells


/usr/local/lib/python3.12/dist-packages/spatialdata/_core/spatialdata.py:184: UserWarning: The table is annotating 'cell_circles', which is not present in the SpatialData object.
  self.validate_table_in_spatialdata(v)


In [8]:
sdata_subset

SpatialData object
├── Images
│     └── 'morphology_focus': DataTree[cyx] (4, 30420, 54160), (4, 15210, 27080), (4, 7605, 13540), (4, 3802, 6770), (4, 1901, 3385)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (30420, 54160), (15210, 27080), (7605, 13540), (3802, 6770), (1901, 3385)
│     └── 'nucleus_labels': DataTree[yx] (30420, 54160), (15210, 27080), (7605, 13540), (3802, 6770), (1901, 3385)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 13) (3D points)
└── Tables
      └── 'table': AnnData (9650, 5006)
with coordinate systems:
    ▸ 'global', with elements:
        morphology_focus (Images), cell_labels (Labels), nucleus_labels (Labels), transcripts (Points)

## 2. Explore the SpatialData object

Now that we have loaded the `SpatialData` object, we can explore its contents. A `SpatialData` object can contain multiple elements, such as images, tables, and shapes.

In [10]:
print(f"Images: {list(sdata_subset.images.keys())}")
print(f"Tables: {list(sdata_subset.tables.keys())}")
print(f"Shapes: {list(sdata_subset.shapes.keys())}")

Images: ['morphology_focus']
Tables: ['table']
Shapes: []


In [18]:
sdata = sdata_subset

### 2.1. Gene Expression Matrix

The gene expression data is stored in the `table` element. We can access it and convert it to a torch tensor.

In [19]:
sdata.tables['table'].layers['raw'] = sdata.tables['table'].X.copy()

In [20]:
sc.pp.normalize_total(sdata.tables['table'])
sc.pp.log1p(sdata.tables['table'])

/usr/local/lib/python3.12/dist-packages/legacy_api_wrap/__init__.py:82: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


In [21]:
print(sdata.tables['table'].X)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1512225 stored elements and shape (9650, 5006)>
  Coords	Values
  (0, 72)	1.147883415222168
  (0, 83)	0.7303261756896973
  (0, 100)	0.7303261756896973
  (0, 151)	0.7303261756896973
  (0, 212)	0.7303261756896973
  (0, 213)	0.7303261756896973
  (0, 261)	0.7303261756896973
  (0, 265)	0.7303261756896973
  (0, 302)	1.147883415222168
  (0, 371)	0.7303261756896973
  (0, 402)	1.147883415222168
  (0, 545)	1.147883415222168
  (0, 555)	0.7303261756896973
  (0, 576)	0.7303261756896973
  (0, 612)	0.7303261756896973
  (0, 647)	0.7303261756896973
  (0, 723)	0.7303261756896973
  (0, 731)	0.7303261756896973
  (0, 763)	0.7303261756896973
  (0, 851)	0.7303261756896973
  (0, 885)	0.7303261756896973
  (0, 887)	0.7303261756896973
  (0, 991)	0.7303261756896973
  (0, 1003)	0.7303261756896973
  (0, 1097)	0.7303261756896973
  :	:
  (9649, 4343)	0.3356693685054779
  (9649, 4348)	0.3356693685054779
  (9649, 4368)	0.3356693685054779
  (9649, 4391)	0.586

In [22]:
expression_matrix = sdata.tables['table'].X.toarray()
print(f"Expression matrix shape: {expression_matrix.shape}")

Expression matrix shape: (9650, 5006)


### 2.2. Spatial Graph

To build the spatial graph, we need the spatial coordinates of the cells. These are stored in the `shapes` element.

In [23]:
adata = sdata.tables["table"]

# Create the cell_coords DataFrame
cell_coords = gpd.GeoDataFrame({
    "x": adata.obs["x"],
    "y": adata.obs["y"]
})

cell_coords

,x,y
cell_id,,
16897,11112.040039,164.095230
170600,6590.570801,3409.162598
168055,6648.562012,1375.255371
7639,2154.809814,3031.503662
107784,1049.577637,2935.302490
...,...,...
118436,8748.960938,284.275970
94621,2485.415527,2629.614258
6754,2133.416992,2430.442383


In [24]:
# --- Build the k-NN graph (this returns a scipy.sparse.csr_matrix) ---
n_neighbors = 6
spatial_graph_sparse_scipy = kneighbors_graph(cell_coords, n_neighbors=n_neighbors, mode='connectivity', include_self=False)

print("Built k-NN graph in scipy sparse format.")

# --- Convert the scipy sparse matrix to a PyTorch sparse tensor ---

# 1. Get the matrix in COOrdinate format, which is easy to work with
coo = spatial_graph_sparse_scipy.tocoo()

# 2. Create the indices for the sparse tensor (the coordinates of the non-zero values)
indices = torch.from_numpy(np.vstack((coo.row, coo.col))).long()

# 3. Create the values for the sparse tensor (the non-zero values themselves)
values = torch.from_numpy(coo.data).float()

# 4. Get the shape of the original matrix
shape = torch.Size(coo.shape)

# 5. Create the PyTorch sparse tensor
spatial_graph_sparse_tensor = torch.sparse_coo_tensor(indices, values, shape)

Built k-NN graph in scipy sparse format.


In [25]:
spatial_graph_sparse_tensor

tensor(indices=tensor([[   0,    0,    0,  ..., 9649, 9649, 9649],
                       [8931, 4017, 4460,  ..., 8685, 1257, 4360]]),
       values=tensor([1., 1., 1.,  ..., 1., 1., 1.]),
       size=(9650, 9650), nnz=57900, layout=torch.sparse_coo)

Now we can build a spatial graph. A common approach is to use a k-nearest neighbors (k-NN) graph.

### 2.3. Gene Bins

To create the gene bins, we need information about the genomic location of each gene. This information is often stored in the `sdata.table.var` dataframe.

In [26]:
sdata.tables['table'].var.head()

,gene_ids,feature_types,genome
A2ML1,ENSG00000166535,Gene Expression,Unknown
AAMP,ENSG00000127837,Gene Expression,Unknown
AAR2,ENSG00000131043,Gene Expression,Unknown
AARSD1,ENSG00000266967,Gene Expression,Unknown
ABAT,ENSG00000183044,Gene Expression,Unknown


In [27]:
cnv.io.genomic_position_from_biomart(sdata.tables['table'], adata_gene_id="gene_ids" ,species="hsapiens")

In [28]:
sdata.tables['table'].var.head()

,gene_ids,feature_types,genome,ensembl_gene_id,start,end,chromosome
A2ML1,ENSG00000166535,Gene Expression,Unknown,ENSG00000166535,8822621,8887001,chr12
AAMP,ENSG00000127837,Gene Expression,Unknown,ENSG00000127837,218264125,218270178,chr2
AAR2,ENSG00000131043,Gene Expression,Unknown,ENSG00000131043,36236131,36270918,chr20
AARSD1,ENSG00000266967,Gene Expression,Unknown,ENSG00000266967,42950431,42964498,chr17
ABAT,ENSG00000183044,Gene Expression,Unknown,ENSG00000183044,8674596,8784575,chr16


Now we need to define a strategy to bin the genes. For example, we can bin them by chromosome.

In [29]:
import pandas as pd

def create_gene_bins(gene_info_df, genes_per_bin=100):
    """
    Creates genomic bins with a fixed number of genes.

    Args:
        gene_info_df (pd.DataFrame): DataFrame with gene information, must contain 'chromosome', 'start', and 'end' columns.
        genes_per_bin (int): The number of genes to include in each bin.

    Returns:
        dict: A dictionary where keys are bin names (e.g., 'chr1_bin1') and
              values are lists of integer indices of the genes in that bin.
    """

    # --- 1. Prepare and sort the gene information ---

    # Make a copy to avoid modifying the original dataframe
    gene_info = gene_info_df.copy()

    # Drop genes with no chromosome information
    gene_info = gene_info.dropna(subset=['chromosome', 'start'])

    # Ensure chromosome names are strings and sort them naturally (e.g., chr1, chr2, ..., chrX)
    gene_info['chromosome'] = gene_info['chromosome'].astype(str)

    # Create a categorical type for natural sorting of chromosome names
    chrom_order = [f'chr{i}' for i in range(1, 23)] + ['chrX', 'chrY']
    gene_info['chromosome'] = pd.Categorical(gene_info['chromosome'], categories=chrom_order, ordered=True)

    # Sort genes by chromosome and start position
    sorted_genes = gene_info.sort_values(['chromosome', 'start'])

    # --- 2. Create the bins ---

    gene_bins = {}
    current_chrom = None
    bin_counter = 0

    for i in range(0, len(sorted_genes), genes_per_bin):
        bin_df = sorted_genes.iloc[i:i+genes_per_bin]

        # Get the chromosome of the first gene in the bin
        chrom = bin_df['chromosome'].iloc[0]

        # Reset bin counter for each new chromosome
        if chrom != current_chrom:
            bin_counter = 0
            current_chrom = chrom

        bin_name = f"{chrom}_bin{bin_counter}"

        # Get the integer indices of the genes in the bin
        # These indices correspond to the rows in the original expression matrix
        gene_indices = bin_df.index.map(lambda x: gene_info_df.index.get_loc(x)).tolist()

        gene_bins[bin_name] = gene_indices
        bin_counter += 1

    return gene_bins

In [30]:
gene_info_df = sdata.tables['table'].var

# Create the gene bins
genes_per_bin = 20  # This is a hyperparameter you can tune
gene_bins = create_gene_bins(gene_info_df, genes_per_bin=genes_per_bin)

# --- You can inspect the first few bins ---
print(f"Created {len(gene_bins)} bins with approximately {genes_per_bin} genes per bin.")
for i, (bin_name, genes) in enumerate(gene_bins.items()):
    if i >= 5:
        break
    print(f"{bin_name}: Contains {len(genes)} genes")

Created 251 bins with approximately 20 genes per bin.
chr1_bin0: Contains 20 genes
chr1_bin1: Contains 20 genes
chr1_bin2: Contains 20 genes
chr1_bin3: Contains 20 genes
chr1_bin4: Contains 20 genes


## Save the Processed Data

Finally, we will save the processed data (gene expression matrix, spatial graph, and gene bins) to files that can be used as input for the ClearCNV model.


In [31]:
# --- Make sure all data is in the correct format ---
# Convert expression matrix to tensor if it's a numpy array
if not isinstance(expression_matrix, torch.Tensor):
    expression_matrix = torch.from_numpy(expression_matrix).float()

# --- Create a dictionary to hold all the data ---
processed_data = {
    'expression_matrix': expression_matrix,
    'spatial_graph': spatial_graph_sparse_tensor,
    'gene_bins': gene_bins
}

# --- Define the output path and save the data ---
output_dir = "../datasets/processed"
output_path = os.path.join(output_dir, "processed_data.pt")

torch.save(processed_data, output_path)

print(f"Processed data saved to: {output_path}")

Processed data saved to: ../datasets/processed/processed_data.pt
